In [1]:
import pandas as pd

In [2]:
# -------------------------------
# 1. Load the centroid data (centroids_with_hybas_ID.csv)
# -------------------------------
centroids_df = pd.read_csv(r"C:\Users\SID-DRW\Downloads\centroids_with_hybas_ID.csv")

# -------------------------------
# 2. Load the HYBAS upstream basin information (Brahmaputra_HYBAS_level7_info.csv)
# -------------------------------
basin_info_df = pd.read_csv(r"C:\Users\SID-DRW\Downloads\Brahmaputra_HYBAS_level7_info.csv")

In [3]:
# Ensure numeric type for IDs
basin_info_df["HYBAS_ID"] = basin_info_df["HYBAS_ID"].astype(int)
basin_info_df["NEXT_DOWN"] = basin_info_df["NEXT_DOWN"].astype(int)

# Build lookup for downstream relationships
downstream_map = basin_info_df.set_index("HYBAS_ID")["NEXT_DOWN"].to_dict()

# Invert downstream map: create map from parent → list of immediate children
upstream_map = {}
for child, parent in downstream_map.items():
    upstream_map.setdefault(parent, []).append(child)

# Function to recursively collect all upstream basins
def collect_upstream(basin_id, upstream_map, visited=None):
    if visited is None:
        visited = set()
    # Add current basin
    visited.add(basin_id)

    # Find direct upstream basins
    children = upstream_map.get(basin_id, [])
    for child in children:
        if child not in visited:
            collect_upstream(child, upstream_map, visited)

    return visited

# ----------------------------
# 3) Compute merged upstream lists
# ----------------------------
results = []

for _, row in centroids_df.iterrows():
    cid = row["centroid_x"]
    cy = row["centroid_y"]
    label_r = row.get("r", None)     # keep r label
    hybas_id = int(row["HYBAS_ID"])  # centroid's basin

    # Collect all upstream basins including itself
    upstream_ids = collect_upstream(hybas_id, upstream_map)

    results.append({
        "centroid_x": cid,
        "centroid_y": cy,
        "r": label_r,
        "centroid_hybas_id": hybas_id,
        "upstream_hybas_ids": sorted(list(upstream_ids))
    })

# ----------------------------
# 4) Put into a DataFrame
# ----------------------------
merged_df = pd.DataFrame(results)

# Save CSV
output_path = "C:\\Users\\SID-DRW\\Downloads\\centroids_upstream_basins_merged.csv"
merged_df.to_csv(output_path, index=False)
print(f"Saved merged upstream basin list to: {output_path}")

# Quick preview
print(merged_df.head())

Saved merged upstream basin list to: C:\Users\SID-DRW\Downloads\centroids_upstream_basins_merged.csv
   centroid_x  centroid_y   r  centroid_hybas_id  \
0   89.677827   25.244097  r2         4070928420   
1   89.697230   25.354590  r3         4070928420   
2   89.714478   25.481252  r4         4070910920   
3   89.655459   25.172950  r1         4070928420   
4   90.299281   26.158492  r9         4070892310   

                                  upstream_hybas_ids  
0  [4070061740, 4070066980, 4070747320, 407074754...  
1  [4070061740, 4070066980, 4070747320, 407074754...  
2  [4070061740, 4070066980, 4070747320, 407074754...  
3  [4070061740, 4070066980, 4070747320, 407074754...  
4  [4070061740, 4070066980, 4070747320, 407074754...  
